In [16]:
import torch 
import torchvision
from torch.utils.data import DataLoader,Dataset,WeightedRandomSampler
import nibabel as nib
import pandas as   pd
from pathlib import Path
import wandb
import tqdm

import torch.optim as optim
from collections import Counter
import numpy as np
import sys
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.append(str(root))

from MST.mst.data.datamodules.datamodule import DataModule
from MST.mst.data.datamodules.datamodule import DataModule
from MST.mst.data.datasets.dataset_3d_mrnet import MRNet_Dataset3D
from MST.mst.data.datasets.augmentations.augmentations_3d import SubjectToTensor,ImageOrSubjectToTensor, ImageToTensor,ZNormalization, RescaleIntensity,EnsureShapeMultiple
from MST.mst.models.dino import DinoV2ClassifierSlice

In [17]:
split=pd.read_csv(r'D:\Project\ACL\my_tool\MST_test_TN_data\preprocessed\splits\split.csv')
split.head()

,ID,PatientKey,RawLabel,AclLabelName,abnormal,acl,meniscus,Plane,Folder,Split,Fold,NrrdPath,SegPath,NiftiPath
0,0,2025_14_11_Thanh/Cai Kim Loc_25210566,Dut_ban_phan_1_3_giua,acl_injury,1,1,0,sagittal,valid/,test,0,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\C...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\C...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
1,1,2025_14_11_Thanh/Ha Van Chau_20250507,Dut_hoan_toan_1_3_giua,acl_injury,1,1,0,sagittal,valid/,test,0,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\H...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\H...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
2,2,2025_14_11_Thanh/Le Thi Thuy_25193052,Dut_ban_phan_1_3_duoi,acl_injury,1,1,0,sagittal,valid/,test,0,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
3,3,2025_14_11_Thanh/Le Van Chung_25938957,Dut_ban_phan_1_3_duoi,acl_injury,1,1,0,sagittal,valid/,test,0,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\L...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...
4,4,2025_14_11_Thanh/Nguyen Hoang Gia Tuan_20250616,Dut_hoan_toan_1_3_giua,acl_injury,1,1,0,sagittal,valid/,test,0,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\N...,D:\Project\ACL\my_tool\data\2025_14_11_Thanh\N...,D:\Project\ACL\my_tool\MST_test_TN_data\prepro...


In [18]:
path_root='D:\Project\ACL\my_tool\MST_test_TN_data'
train_dataset=MRNet_Dataset3D(path_root=path_root,
                              split='train',flip=True,random_rotate=True,noise=True)
val_dataset = MRNet_Dataset3D(path_root=path_root,
    split='val',
    flip=False,
    random_rotate=False,
    noise=False
)

test_dataset = MRNet_Dataset3D(path_root=path_root,
    split='test',
    flip=False,
    random_rotate=False,
    noise=False
)

In [19]:
for i in range(10):
    print(test_dataset[i]['source'].shape)
print(len(test_dataset))

d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)


torch.Size([1, 32, 224, 224])


d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 224, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)


torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 224, 224])


d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (224, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)


torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 224, 224])
166


In [20]:
datamodule=DataModule(ds_train=train_dataset,
                      ds_test=test_dataset,
                      ds_val=val_dataset,batch_size=16,batch_size_test=16,batch_size_val=16)

test_dataloader=datamodule.test_dataloader()

In [21]:
model = DinoV2ClassifierSlice(
    in_ch=1,
    out_ch=2,
    pretrained=True,
    model_size='s',
    use_registers=False,
    use_bottleneck=False,
    use_slice_pos_emb=True,
    slice_fusion='transformer',
    freeze=True,
    optimizer_kwargs={'lr': 1e-4, 'weight_decay': 1e-4},
)

Using cache found in C:\Users\hp/.cache\torch\hub\facebookresearch_dinov2_main
d:\Project\ACL\.venv\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer was not TransformerEncoderLayer
  warnings.warn(


In [27]:
from pathlib import Path
import torch
import tqdm
import numpy as np



class Evaluator:
    def __init__(
        self,
        model,
        test_loader,
        checkpoint_folder: Path,
        device="cuda",
        criterion=torch.nn.CrossEntropyLoss(),
    ):
        self.test_loader = test_loader
        self.checkpoint_folder = checkpoint_folder
        self.model = model
        self.device = device
        self.criterion = criterion
        self.batch_size = test_loader.batch_size

    def loading_model_state_dict(self, model, statedict):
        model.load_state_dict(statedict)
        return model

    def evaluate(self):
        inference_result = []

        for pth_file in self.checkpoint_folder.glob("*.pt"):
            print(f"Loading checkpoint: {pth_file.name}")

            checkpoint = torch.load(pth_file, map_location=self.device)

            # if checkpoint is a full training checkpoint
            if "model_state_dict" in checkpoint:
                state_dict = checkpoint["model_state_dict"]
            else:
                state_dict = checkpoint  # in case some files are pure state_dict files

            self.model = self.loading_model_state_dict(self.model, state_dict)
            self.model = self.model.to(self.device)
            self.model.eval()
            # initialize metric accumulators
            running_loss = 0.0
            correct = 0
            total = 0
            all_preds = []
            all_labels = []

            loop = tqdm.tqdm(self.test_loader, desc=f"Inferencing {pth_file.name}")
            print(f"Using device: {self.device}")

            with torch.no_grad():
                for batch in loop:
                    images = batch["source"].to(self.device)
                    labels = batch["target"].to(self.device)

                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)

                    batch_size = images.size(0)
                    running_loss += loss.item() * batch_size

                    preds = outputs.argmax(dim=1)
                    total += batch_size
                    correct += (preds == labels).sum().item()

                    all_preds.append(preds.cpu())
                    all_labels.append(labels.cpu())

            all_preds = torch.cat(all_preds).numpy()
            all_labels = torch.cat(all_labels).numpy()

            avg_loss = running_loss / total
            acc = correct / total
            macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
            weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
            balanced_acc = balanced_accuracy_score(all_labels, all_preds)
            cm = confusion_matrix(all_labels, all_preds)

            checkpoint_inference_result = {
                "checkpoint": pth_file.name,
                "loss": avg_loss,
                "accuracy": acc,
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1,
                "balanced_acc": balanced_acc,
                "cm": cm,
            }

            inference_result.append(checkpoint_inference_result)

        return inference_result

    def visualize_result(self, result, class_names=None, save_dir=None, figsize=(8, 6)):
        """
        Visualize evaluation results for every checkpoint.

        Args:
            result (list[dict]): output from self.evaluate()
            class_names (list[str] | None): names of classes for confusion matrix axes
            save_dir (str | Path | None): folder to save figures; if None, just show them
            figsize (tuple): figure size for confusion matrix
        """
        if len(result) == 0:
            print("No result to visualize.")
            return

        if save_dir is not None:
            save_dir = Path(save_dir)
            save_dir.mkdir(parents=True, exist_ok=True)

        # -------- Summary bar chart across all checkpoints --------
        checkpoint_names = [r["checkpoint"] for r in result]
        macro_f1s = [r["macro_f1"] for r in result]
        weighted_f1s = [r["weighted_f1"] for r in result]
        balanced_accs = [r["balanced_acc"] for r in result]
        accuracies = [r.get("accuracy", None) for r in result]

        x = np.arange(len(checkpoint_names))
        width = 0.2

        plt.figure(figsize=(max(10, len(checkpoint_names) * 1.5), 6))
        plt.bar(x - 1.5 * width, macro_f1s, width, label="Macro F1")
        plt.bar(x - 0.5 * width, weighted_f1s, width, label="Weighted F1")
        plt.bar(x + 0.5 * width, balanced_accs, width, label="Balanced Acc")

        if all(a is not None for a in accuracies):
            plt.bar(x + 1.5 * width, accuracies, width, label="Accuracy")

        plt.xticks(x, checkpoint_names, rotation=45, ha="right")
        plt.ylim(0, 1.0)
        plt.ylabel("Score")
        plt.title("Checkpoint Performance Comparison")
        plt.legend()
        plt.tight_layout()

        if save_dir is not None:
            plt.savefig(save_dir / "checkpoint_metric_comparison.png", dpi=300, bbox_inches="tight")
            plt.close()
        else:
            plt.show()

        # -------- Confusion matrix for each checkpoint --------
        for r in result:
            cm = r["cm"]
            checkpoint_name = r["checkpoint"]

            fig, ax = plt.subplots(figsize=figsize)
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm,
                display_labels=class_names if class_names is not None else None,
            )
            disp.plot(ax=ax, cmap="Blues", colorbar=False)

            ax.set_title(
                f"{checkpoint_name}\n"
                f"Macro F1={r['macro_f1']:.4f} | "
                f"Weighted F1={r['weighted_f1']:.4f} | "
                f"Balanced Acc={r['balanced_acc']:.4f}"
            )
            plt.tight_layout()

            if save_dir is not None:
                safe_name = checkpoint_name.replace(".pth", "").replace("/", "_").replace("\\", "_")
                plt.savefig(save_dir / f"{safe_name}_confusion_matrix.png", dpi=300, bbox_inches="tight")
                plt.close()
            else:
                plt.show()

In [28]:
evaluator = Evaluator(
    model=model,
    test_loader=test_dataloader,
    checkpoint_folder=Path("./checkpoints"),
    device="cuda"
)

results = evaluator.evaluate()

evaluator.visualize_result(
    result=results,
    class_names=["class_0", "class_1"],   # change this to your real class names
    save_dir="./eval_figures"
)

Loading checkpoint: best_generous-sweep-5_yg4ngc5e.pt


Inferencing best_generous-sweep-5_yg4ngc5e.pt:   0%|          | 0/11 [00:00<?, ?it/s]d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)


Using device: cuda


d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 224, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (224, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Inferencing best_generous-sweep-5_yg4ngc5e.pt: 100%|██████████| 11/11 [01:59<00:00, 10.86s/it]


Loading checkpoint: best_hopeful-sweep-2_9sislb46.pt


...

d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 224, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (224, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Inferencing best_serene-sweep-4_bi4er13r.pt: 100%|██████████| 11/11 [01:53<00:00, 10.35s/it]


Loading checkpoint: best_vocal-sweep-2_211f7gd9.pt


Inferencing best_vocal-sweep-2_211f7gd9.pt:   0%|          | 0/11 [00:00<?, ?it/s]

Using device: cuda


d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (225, 224, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
d:\Project\ACL\.venv\Lib\site-packages\torchio\transforms\transform.py:168: RuntimeWarning: Output shape (224, 225, 32) != target shape (np.int64(224), np.int64(224), np.int64(32)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Inferencing best_vocal-sweep-2_211f7gd9.pt: 100%|██████████| 11/11 [01:53<00:00, 10.33s/it]
